# Notebook lecture 7: Lyapunov Theory for a Nonlinear 2-State System

In this notebook, we build an interactive Lyapunov stability demo for a simple nonlinear system with a strict PID controller.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed for 3D projection)
from scipy import linalg
import ipywidgets as widgets
from ipywidgets import FloatSlider, VBox, HBox, HTML

# Prefer interactive backend for sliders + 3D rotation in VS Code/Jupyter.
try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams["figure.figsize"] = (9, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

## System and Lyapunov setup

We use the nonlinear plant

$$
\dot{x}_1 = x_2, \qquad
\dot{x}_2 = -\sin(x_1) - c x_2 + u.
$$

The controller is strict PID around reference $r=0$:

$$
e = r - x_1 = -x_1, \qquad
\dot{z} = e,
$$
$$
u = K_p e + K_i z - K_d x_2.
$$

The plant remains 2-state ($x_1,x_2$), while $z$ is the controller memory state.

For the displayed Lyapunov surface, we compute a **local quadratic candidate** from the linearized augmented closed-loop model at the origin:

$$
A_{cl}^T P + P A_{cl} = -Q,
$$

then plot the slice

$$
V(x_1,x_2) = [x_1\;x_2] P_{2\times 2} [x_1\;x_2]^T
$$

with $z=0$. This makes the surface gain-dependent and pedagogically useful near the equilibrium.

In [ ]:
def augmented_linearized_A(kp, ki, kd, c=0.4):
    """Linearized closed-loop matrix around (x1,x2,z)=(0,0,0), r=0."""
    # x1_dot = x2
    # x2_dot = -(1+kp)*x1 -(c+kd)*x2 + ki*z
    # z_dot  = -x1
    return np.array(
        [
            [0.0, 1.0, 0.0],
            [-(1.0 + kp), -(c + kd), ki],
            [-1.0, 0.0, 0.0],
        ],
        dtype=float,
    )


def local_lyapunov_slice_matrix(kp, ki, kd, c=0.4):
    """Return P2 for V(x1,x2)=x^T P2 x, plus status message."""
    A = augmented_linearized_A(kp, ki, kd, c=c)
    eigvals = np.linalg.eigvals(A)

    if np.max(np.real(eigvals)) >= -1e-6:
        return None, "Linearized closed-loop is not asymptotically stable for these gains."

    Q = np.diag([3.0, 2.0, 0.8])
    # Solve A^T P + P A = -Q.
    P = linalg.solve_continuous_lyapunov(A.T, -Q)
    P = 0.5 * (P + P.T)

    if np.any(np.linalg.eigvals(P) <= 1e-10):
        return None, "Lyapunov matrix P is not positive definite for these gains."

    P2 = P[:2, :2]
    if np.any(np.linalg.eigvals(P2) <= 1e-10):
        return None, "Projected matrix P2 is not positive definite in (x1,x2)."

    return P2, "OK"


def simulate_nonlinear_pid(
    kp,
    ki,
    kd,
    x10,
    x20,
    z0,
    c=0.4,
    dt=0.01,
    T=12.0,
):
    """Simulate nonlinear closed-loop system with simple forward Euler."""
    n = int(np.floor(T / dt)) + 1
    t = np.linspace(0.0, dt * (n - 1), n)

    x1 = np.zeros(n)
    x2 = np.zeros(n)
    z = np.zeros(n)
    x1[0], x2[0], z[0] = x10, x20, z0

    diverged = False
    for k in range(n - 1):
        e = -x1[k]
        u = kp * e + ki * z[k] - kd * x2[k]

        x1_dot = x2[k]
        x2_dot = -np.sin(x1[k]) - c * x2[k] + u
        z_dot = e

        x1[k + 1] = x1[k] + dt * x1_dot
        x2[k + 1] = x2[k] + dt * x2_dot
        z[k + 1] = z[k] + dt * z_dot

        if np.abs(x1[k + 1]) > 15 or np.abs(x2[k + 1]) > 15 or np.abs(z[k + 1]) > 15:
            x1[k + 1 :] = np.nan
            x2[k + 1 :] = np.nan
            z[k + 1 :] = np.nan
            diverged = True
            break

    return t, x1, x2, z, diverged


def V_from_P2(P2, x1, x2):
    return P2[0, 0] * x1**2 + 2.0 * P2[0, 1] * x1 * x2 + P2[1, 1] * x2**2

In [ ]:
output = widgets.Output()
status = HTML(value="<b>Status:</b> Ready")

kp_slider = FloatSlider(description="Kp", min=0.0, max=8.0, step=0.1, value=2.0, continuous_update=False)
ki_slider = FloatSlider(description="Ki", min=0.0, max=5.0, step=0.1, value=1.0, continuous_update=False)
kd_slider = FloatSlider(description="Kd", min=0.0, max=5.0, step=0.1, value=1.2, continuous_update=False)

x10_slider = FloatSlider(description="x1(0)", min=-2.5, max=2.5, step=0.05, value=1.2, continuous_update=False)
x20_slider = FloatSlider(description="x2(0)", min=-2.5, max=2.5, step=0.05, value=0.0, continuous_update=False)
z0_slider = FloatSlider(description="z(0)", min=-2.0, max=2.0, step=0.05, value=0.0, continuous_update=False)


def refresh_plot(_=None):
    kp = kp_slider.value
    ki = ki_slider.value
    kd = kd_slider.value
    x10 = x10_slider.value
    x20 = x20_slider.value
    z0 = z0_slider.value

    with output:
        output.clear_output(wait=True)

        P2, msg = local_lyapunov_slice_matrix(kp, ki, kd)

        fig = plt.figure(figsize=(17, 5.8), constrained_layout=True)
        ax3d = fig.add_subplot(1, 3, 1, projection="3d")
        ax_open = fig.add_subplot(1, 3, 2)
        ax_closed = fig.add_subplot(1, 3, 3)

        # Open-loop (no control): kp=ki=kd=0 makes u=0.
        t_ol, x1_ol, x2_ol, z_ol, div_ol = simulate_nonlinear_pid(0.0, 0.0, 0.0, x10, x20, 0.0)
        valid_ol = np.isfinite(x1_ol) & np.isfinite(x2_ol)

        # Closed-loop (with PID control).
        t_cl, x1_cl, x2_cl, z_cl, div_cl = simulate_nonlinear_pid(kp, ki, kd, x10, x20, z0)
        valid_cl = np.isfinite(x1_cl) & np.isfinite(x2_cl)

        if P2 is None:
            ax3d.text2D(
                0.05,
                0.92,
                "No valid local Lyapunov surface for this gain set.\nTry increasing damping or lowering integral action.",
                transform=ax3d.transAxes,
                fontsize=10,
                color="crimson",
            )
            ax3d.set_title("Lyapunov Surface Unavailable")
            ax3d.set_xlabel("x1")
            ax3d.set_ylabel("x2")
            ax3d.set_zlabel("V")
            status.value = f"<b>Status:</b> {msg}"
        else:
            g = np.linspace(-2.2, 2.2, 80)
            X1, X2 = np.meshgrid(g, g)
            Vsurf = V_from_P2(P2, X1, X2)

            surf = ax3d.plot_surface(X1, X2, Vsurf, cmap="viridis", alpha=0.85, linewidth=0, antialiased=True)
            fig.colorbar(surf, ax=ax3d, shrink=0.72, pad=0.07, label="V(x1,x2)")

            if np.count_nonzero(valid_cl) > 2:
                Vtraj = V_from_P2(P2, x1_cl[valid_cl], x2_cl[valid_cl])
                ax3d.plot(x1_cl[valid_cl], x2_cl[valid_cl], Vtraj, color="crimson", linewidth=2.6, label="Closed-loop trajectory")
                ax3d.scatter([x1_cl[valid_cl][0]], [x2_cl[valid_cl][0]], [Vtraj[0]], color="black", s=35, label="Start")

            ax3d.scatter([0.0], [0.0], [0.0], color="white", edgecolor="black", s=42, label="Origin")
            ax3d.set_title("3D Local Lyapunov Surface + Trajectory")
            ax3d.set_xlabel("x1")
            ax3d.set_ylabel("x2")
            ax3d.set_zlabel("V(x1,x2)")
            ax3d.view_init(elev=30, azim=-55)
            ax3d.legend(loc="upper right")

            if div_cl:
                status.value = "<b>Status:</b> Controlled simulation diverged before horizon."
            else:
                status.value = "<b>Status:</b> OK"

        # 2D evolution without control.
        if np.count_nonzero(valid_ol) > 1:
            ax_open.plot(x1_ol[valid_ol], x2_ol[valid_ol], color="tab:gray", linewidth=2.2, label="Open-loop")
            ax_open.scatter([x1_ol[valid_ol][0]], [x2_ol[valid_ol][0]], color="black", s=26, zorder=3, label="Start")
        ax_open.scatter([0.0], [0.0], color="tab:green", s=28, zorder=3, label="Origin")
        ax_open.set_title("2D Evolution Without Control")
        ax_open.set_xlabel("x1")
        ax_open.set_ylabel("x2")
        ax_open.set_xlim(-2.6, 2.6)
        ax_open.set_ylim(-2.6, 2.6)
        ax_open.set_aspect("equal", adjustable="box")
        ax_open.grid(True, alpha=0.35)
        if div_ol:
            ax_open.text(0.03, 0.96, "Diverged", transform=ax_open.transAxes, va="top", color="crimson")
        ax_open.legend(loc="upper right")

        # 2D evolution with control (top-down phase view).
        if np.count_nonzero(valid_cl) > 1:
            ax_closed.plot(x1_cl[valid_cl], x2_cl[valid_cl], color="crimson", linewidth=2.2, label="Closed-loop")
            ax_closed.scatter([x1_cl[valid_cl][0]], [x2_cl[valid_cl][0]], color="black", s=26, zorder=3, label="Start")
        ax_closed.scatter([0.0], [0.0], color="tab:green", s=28, zorder=3, label="Origin")
        ax_closed.set_title("2D Evolution With PID (Top View)")
        ax_closed.set_xlabel("x1")
        ax_closed.set_ylabel("x2")
        ax_closed.set_xlim(-2.6, 2.6)
        ax_closed.set_ylim(-2.6, 2.6)
        ax_closed.set_aspect("equal", adjustable="box")
        ax_closed.grid(True, alpha=0.35)
        if div_cl:
            ax_closed.text(0.03, 0.96, "Diverged", transform=ax_closed.transAxes, va="top", color="crimson")
        ax_closed.legend(loc="upper right")

        plt.show()


for w in [kp_slider, ki_slider, kd_slider, x10_slider, x20_slider, z0_slider]:
    w.observe(refresh_plot, names="value")

controls = VBox(
    [
        HTML("<h4>PID gains</h4>"),
        HBox([kp_slider, ki_slider, kd_slider]),
        HTML("<h4>Initial conditions</h4>"),
        HBox([x10_slider, x20_slider, z0_slider]),
        status,
    ]
)

display(controls, output)
refresh_plot()

## Try-this questions

1. Increase $K_d$ while keeping $K_p, K_i$ fixed. How does the trajectory shape and settling trend change?
2. Increase $K_i$ too much. When does the local model lose validity or stability?
3. Start farther from the origin with the same gains. Does the trajectory still follow the local Lyapunov geometry?
4. Compare two gain sets with similar settling speed but different overshoot. How is the 3D path different over the same surface?

## Exercise: 3D Pure-Pursuit Rendezvous (Nonlinear Dynamics)

In this second exercise, a pursuer dot tries to rendezvous with a moving target dot in 3D using a **pure pursuit guidance law**.

In [ ]:
def target_trajectory(t, vx=1.2, y_amp=1.5, z_amp=0.8, omega=0.45, z0=0.5):
    """Straight-line drift in x with mild sinusoidal maneuver in y and z."""
    x = vx * t
    y = y_amp * np.sin(omega * t)
    z = z0 + z_amp * np.sin(0.6 * omega * t + 0.3)

    vx_t = np.full_like(t, vx)
    vy_t = y_amp * omega * np.cos(omega * t)
    vz_t = z_amp * 0.6 * omega * np.cos(0.6 * omega * t + 0.3)

    p_t = np.column_stack((x, y, z))
    v_t = np.column_stack((vx_t, vy_t, vz_t))
    return p_t, v_t


def _limited_turn_direction(current_dir, desired_dir, max_turn_rad):
    """Rotate current_dir toward desired_dir by at most max_turn_rad."""
    dot = np.clip(np.dot(current_dir, desired_dir), -1.0, 1.0)
    angle = np.arccos(dot)

    if angle <= max_turn_rad:
        return desired_dir

    tangent = desired_dir - dot * current_dir
    nrm = np.linalg.norm(tangent)
    if nrm < 1e-10:
        return current_dir

    tangent /= nrm
    return np.cos(max_turn_rad) * current_dir + np.sin(max_turn_rad) * tangent


def simulate_pure_pursuit(
    k_guidance=3.0,
    k_drag=0.06,
    max_turn_deg=40.0,
    speed0=1.4,
    p0=(-4.0, -2.0, -0.7),
    target_vx=1.2,
    y_amp=1.5,
    z_amp=0.8,
    omega=0.45,
    dt=0.04,
    T=24.0,
    capture_radius=0.25,
):
    n = int(np.floor(T / dt)) + 1
    t = np.linspace(0.0, dt * (n - 1), n)

    p_t, v_t = target_trajectory(t, vx=target_vx, y_amp=y_amp, z_amp=z_amp, omega=omega)

    p_p = np.zeros((n, 3))
    v_p = np.zeros((n, 3))
    p_p[0] = np.array(p0, dtype=float)

    initial_los = p_t[0] - p_p[0]
    los0_norm = np.linalg.norm(initial_los)
    if los0_norm < 1e-10:
        initial_los = np.array([1.0, 0.0, 0.0])
    else:
        initial_los = initial_los / los0_norm

    v_p[0] = speed0 * initial_los

    los_angle_deg = np.zeros(n)
    distance = np.zeros(n)

    intercepted = False
    t_intercept = None
    idx_end = n - 1

    for k in range(n - 1):
        rel = p_t[k] - p_p[k]
        dist = np.linalg.norm(rel)
        distance[k] = dist

        if dist < capture_radius:
            intercepted = True
            t_intercept = t[k]
            idx_end = k
            break

        los_dir = rel / max(dist, 1e-10)

        speed = np.linalg.norm(v_p[k])
        if speed < 1e-8:
            current_dir = los_dir
            speed = 0.3
        else:
            current_dir = v_p[k] / speed

        max_turn = np.deg2rad(max_turn_deg) * dt
        cmd_dir = _limited_turn_direction(current_dir, los_dir, max_turn)

        # Guidance drives velocity direction toward cmd_dir, drag opposes motion.
        a_guidance = k_guidance * (speed * cmd_dir - v_p[k])
        a_drag = -k_drag * speed * v_p[k]
        a_total = a_guidance + a_drag

        v_next = v_p[k] + dt * a_total
        speed_next = np.linalg.norm(v_next)
        if speed_next < 0.15:
            v_next = 0.15 * cmd_dir

        p_p[k + 1] = p_p[k] + dt * v_next
        v_p[k + 1] = v_next

        vhat = v_p[k] / max(np.linalg.norm(v_p[k]), 1e-10)
        los_angle_deg[k] = np.rad2deg(np.arccos(np.clip(np.dot(vhat, los_dir), -1.0, 1.0)))

    distance[idx_end:] = np.linalg.norm(p_t[idx_end:] - p_p[idx_end:], axis=1)

    if not intercepted:
        idx_end = n - 1

    out = {
        "t": t[: idx_end + 1],
        "p_target": p_t[: idx_end + 1],
        "v_target": v_t[: idx_end + 1],
        "p_pursuer": p_p[: idx_end + 1],
        "v_pursuer": v_p[: idx_end + 1],
        "distance": distance[: idx_end + 1],
        "los_angle_deg": los_angle_deg[: idx_end + 1],
        "intercepted": intercepted,
        "t_intercept": t_intercept,
        "capture_radius": capture_radius,
    }
    return out

In [ ]:
pp_output = widgets.Output()
pp_status = HTML(value="<b>Pure Pursuit Status:</b> Ready")

pp_k_slider = FloatSlider(description="k_guid", min=0.5, max=8.0, step=0.1, value=3.0, continuous_update=False)
pp_drag_slider = FloatSlider(description="k_drag", min=0.0, max=0.2, step=0.005, value=0.06, continuous_update=False)
pp_turn_slider = FloatSlider(description="turn deg/s", min=8.0, max=100.0, step=1.0, value=40.0, continuous_update=False)
pp_speed_slider = FloatSlider(description="v0", min=0.3, max=3.5, step=0.05, value=1.4, continuous_update=False)

pp_x0_slider = FloatSlider(description="x0", min=-8.0, max=2.0, step=0.2, value=-4.0, continuous_update=False)
pp_y0_slider = FloatSlider(description="y0", min=-4.0, max=4.0, step=0.1, value=-2.0, continuous_update=False)
pp_z0_slider = FloatSlider(description="z0", min=-3.0, max=3.0, step=0.1, value=-0.7, continuous_update=False)

pp_ampy_slider = FloatSlider(description="Ay", min=0.0, max=3.0, step=0.1, value=1.5, continuous_update=False)
pp_ampz_slider = FloatSlider(description="Az", min=0.0, max=2.0, step=0.1, value=0.8, continuous_update=False)
pp_omega_slider = FloatSlider(description="omega", min=0.1, max=1.2, step=0.05, value=0.45, continuous_update=False)

pp_play = widgets.Play(value=0, min=0, max=1, step=1, interval=45, description="Play", disabled=False)
pp_frame = widgets.IntSlider(value=0, min=0, max=1, step=1, description="frame", continuous_update=True)
widgets.jslink((pp_play, "value"), (pp_frame, "value"))

pp_cache = {}


def _render_pursuit_frame(frame_idx):
    t = pp_cache["t"]
    pt = pp_cache["p_target"]
    pp = pp_cache["p_pursuer"]
    dist = pp_cache["distance"]
    los = pp_cache["los_angle_deg"]

    i = int(np.clip(frame_idx, 0, len(t) - 1))

    with pp_output:
        pp_output.clear_output(wait=True)

        fig = plt.figure(figsize=(15, 5.3), constrained_layout=True)
        ax3d = fig.add_subplot(1, 3, 1, projection="3d")
        axd = fig.add_subplot(1, 3, 2)
        axa = fig.add_subplot(1, 3, 3)

        # 3D chase scene
        ax3d.plot(pt[:, 0], pt[:, 1], pt[:, 2], color="tab:blue", alpha=0.35, linewidth=2.0, label="Target full path")
        ax3d.plot(pp[: i + 1, 0], pp[: i + 1, 1], pp[: i + 1, 2], color="crimson", linewidth=2.5, label="Pursuer path")

        ax3d.scatter([pt[i, 0]], [pt[i, 1]], [pt[i, 2]], color="tab:blue", s=45, label="Target")
        ax3d.scatter([pp[i, 0]], [pp[i, 1]], [pp[i, 2]], color="crimson", s=45, label="Pursuer")

        ax3d.plot(
            [pp[i, 0], pt[i, 0]],
            [pp[i, 1], pt[i, 1]],
            [pp[i, 2], pt[i, 2]],
            color="gray",
            linestyle="--",
            linewidth=1.2,
            alpha=0.8,
            label="LOS",
        )

        xyz = np.vstack((pt, pp))
        mins = xyz.min(axis=0)
        maxs = xyz.max(axis=0)
        center = 0.5 * (mins + maxs)
        span = max((maxs - mins).max(), 1.0)
        half = 0.55 * span

        ax3d.set_xlim(center[0] - half, center[0] + half)
        ax3d.set_ylim(center[1] - half, center[1] + half)
        ax3d.set_zlim(center[2] - half, center[2] + half)
        ax3d.set_box_aspect([1, 1, 1])
        ax3d.set_title("3D Pure-Pursuit Rendezvous")
        ax3d.set_xlabel("x")
        ax3d.set_ylabel("y")
        ax3d.set_zlabel("z")
        ax3d.view_init(elev=22, azim=-62)
        ax3d.legend(loc="upper left")

        # Distance metric
        axd.plot(t, dist, color="0.7", linewidth=1.8)
        axd.plot(t[: i + 1], dist[: i + 1], color="crimson", linewidth=2.3)
        axd.axvline(t[i], color="black", linestyle="--", linewidth=1.0)
        axd.axhline(pp_cache["capture_radius"], color="tab:green", linestyle=":", linewidth=1.4, label="capture radius")
        axd.set_title("Distance-to-Target")
        axd.set_xlabel("time [s]")
        axd.set_ylabel("distance")
        axd.grid(True, alpha=0.35)
        axd.legend(loc="upper right")

        # LOS angle metric
        axa.plot(t, los, color="0.7", linewidth=1.8)
        axa.plot(t[: i + 1], los[: i + 1], color="tab:purple", linewidth=2.3)
        axa.axvline(t[i], color="black", linestyle="--", linewidth=1.0)
        axa.set_title("LOS Alignment Error")
        axa.set_xlabel("time [s]")
        axa.set_ylabel("angle [deg]")
        axa.grid(True, alpha=0.35)

        plt.show()


def _recompute_pursuit(_=None):
    data = simulate_pure_pursuit(
        k_guidance=pp_k_slider.value,
        k_drag=pp_drag_slider.value,
        max_turn_deg=pp_turn_slider.value,
        speed0=pp_speed_slider.value,
        p0=(pp_x0_slider.value, pp_y0_slider.value, pp_z0_slider.value),
        y_amp=pp_ampy_slider.value,
        z_amp=pp_ampz_slider.value,
        omega=pp_omega_slider.value,
    )

    pp_cache.clear()
    pp_cache.update(data)

    n = len(pp_cache["t"])
    pp_frame.max = max(n - 1, 1)
    pp_play.max = max(n - 1, 1)
    pp_play.value = 0
    pp_frame.value = 0

    if pp_cache["intercepted"]:
        pp_status.value = f"<b>Pure Pursuit Status:</b> Intercept at t = {pp_cache['t_intercept']:.2f} s"
    else:
        pp_status.value = f"<b>Pure Pursuit Status:</b> No intercept. Final miss distance = {pp_cache['distance'][-1]:.2f}"

    _render_pursuit_frame(0)


def _on_frame_change(change):
    if change["name"] == "value":
        _render_pursuit_frame(change["new"])


for s in [
    pp_k_slider,
    pp_drag_slider,
    pp_turn_slider,
    pp_speed_slider,
    pp_x0_slider,
    pp_y0_slider,
    pp_z0_slider,
    pp_ampy_slider,
    pp_ampz_slider,
    pp_omega_slider,
]:
    s.observe(_recompute_pursuit, names="value")

pp_frame.observe(_on_frame_change, names="value")

pp_controls = VBox(
    [
        HTML("<h4>Guidance and nonlinear dynamics</h4>"),
        HBox([pp_k_slider, pp_drag_slider, pp_turn_slider, pp_speed_slider]),
        HTML("<h4>Pursuer initial condition</h4>"),
        HBox([pp_x0_slider, pp_y0_slider, pp_z0_slider]),
        HTML("<h4>Target maneuver</h4>"),
        HBox([pp_ampy_slider, pp_ampz_slider, pp_omega_slider]),
        HTML("<h4>Animation</h4>"),
        HBox([pp_play, pp_frame]),
        pp_status,
    ]
)

display(pp_controls, pp_output)
_recompute_pursuit()

### Try-this questions (Pure Pursuit)

1. Increase drag $k_{drag}$ while keeping guidance gain fixed. How does the miss distance change?
2. Decrease max turn-rate. At what point can the pursuer no longer keep up with target maneuvering?
3. Increase target maneuver frequency $\omega$. Why can pure pursuit start to lag and overshoot LOS alignment?
4. Compare high guidance gain with low guidance gain. Which one gives faster closure, and which one looks smoother?

## Exercise: Smarter Guidance with Automatic Look-Ahead

This variant aims **ahead** of the blue target dot instead of aiming directly at its current position.

At each time step, the aim point is

$$
p_{aim}(t) = p_{target}(t) + \tau_{lead}(t) \, v_{target}(t),
$$

with an automatic look-ahead time

$$
\tau_{lead}(t) = \mathrm{clip}\!\left(\tau_{gain}\frac{\|p_{target}-p_{pursuer}\|}{\|v_{target}-v_{pursuer}\|+\varepsilon},\; \tau_{min},\; \tau_{max}\right).
$$

So, when the two dots are far apart or closing slowly, the controller looks farther ahead; when they are close or rapidly changing relative motion, look-ahead is reduced.

The gold star in the 3D plot shows the current aim point.

In [ ]:
def simulate_lead_pursuit(
    k_guidance=3.0,
    k_drag=0.06,
    max_turn_deg=40.0,
    speed0=1.4,
    tau_gain=1.0,
    tau_min=0.1,
    tau_max=2.5,
    p0=(-4.0, -2.0, -0.7),
    target_vx=1.2,
    y_amp=1.5,
    z_amp=0.8,
    omega=0.45,
    dt=0.04,
    T=24.0,
    capture_radius=0.25,
):
    """Lead-pursuit with automatic look-ahead based on distance and relative speed."""
    n = int(np.floor(T / dt)) + 1
    t = np.linspace(0.0, dt * (n - 1), n)

    p_t, v_t = target_trajectory(t, vx=target_vx, y_amp=y_amp, z_amp=z_amp, omega=omega)

    p_p = np.zeros((n, 3))
    v_p = np.zeros((n, 3))
    p_aim = np.zeros((n, 3))
    tau_hist = np.zeros(n)
    p_p[0] = np.array(p0, dtype=float)

    init_rel = p_t[0] - p_p[0]
    init_dist = np.linalg.norm(init_rel)
    init_vrel = np.linalg.norm(v_t[0]) + 1e-6
    tau0 = np.clip(tau_gain * init_dist / init_vrel, tau_min, tau_max)
    tau_hist[0] = tau0

    p_aim[0] = p_t[0] + tau0 * v_t[0]
    initial_dir = p_aim[0] - p_p[0]
    initial_norm = np.linalg.norm(initial_dir)
    if initial_norm < 1e-10:
        initial_dir = np.array([1.0, 0.0, 0.0])
    else:
        initial_dir = initial_dir / initial_norm
    v_p[0] = speed0 * initial_dir

    los_angle_deg = np.zeros(n)
    distance = np.zeros(n)

    intercepted = False
    t_intercept = None
    idx_end = n - 1

    for k in range(n - 1):
        rel_target = p_t[k] - p_p[k]
        dist = np.linalg.norm(rel_target)
        distance[k] = dist

        if dist < capture_radius:
            intercepted = True
            t_intercept = t[k]
            idx_end = k
            break

        rel_speed = np.linalg.norm(v_t[k] - v_p[k])
        tau_k = np.clip(tau_gain * dist / (rel_speed + 1e-6), tau_min, tau_max)
        tau_hist[k] = tau_k

        p_aim[k] = p_t[k] + tau_k * v_t[k]
        rel_aim = p_aim[k] - p_p[k]
        aim_dist = np.linalg.norm(rel_aim)
        aim_dir = rel_aim / max(aim_dist, 1e-10)

        speed = np.linalg.norm(v_p[k])
        if speed < 1e-8:
            current_dir = aim_dir
            speed = 0.3
        else:
            current_dir = v_p[k] / speed

        max_turn = np.deg2rad(max_turn_deg) * dt
        cmd_dir = _limited_turn_direction(current_dir, aim_dir, max_turn)

        a_guidance = k_guidance * (speed * cmd_dir - v_p[k])
        a_drag = -k_drag * speed * v_p[k]
        a_total = a_guidance + a_drag

        v_next = v_p[k] + dt * a_total
        speed_next = np.linalg.norm(v_next)
        if speed_next < 0.15:
            v_next = 0.15 * cmd_dir

        p_p[k + 1] = p_p[k] + dt * v_next
        v_p[k + 1] = v_next

        los_dir = rel_target / max(dist, 1e-10)
        vhat = v_p[k] / max(np.linalg.norm(v_p[k]), 1e-10)
        los_angle_deg[k] = np.rad2deg(np.arccos(np.clip(np.dot(vhat, los_dir), -1.0, 1.0)))

    if idx_end > 0:
        tau_hist[idx_end] = tau_hist[idx_end - 1]
    distance[idx_end:] = np.linalg.norm(p_t[idx_end:] - p_p[idx_end:], axis=1)
    p_aim[idx_end:] = p_t[idx_end:] + tau_hist[idx_end] * v_t[idx_end:]

    if not intercepted:
        idx_end = n - 1

    out = {
        "t": t[: idx_end + 1],
        "p_target": p_t[: idx_end + 1],
        "v_target": v_t[: idx_end + 1],
        "p_pursuer": p_p[: idx_end + 1],
        "p_aim": p_aim[: idx_end + 1],
        "tau_lead": tau_hist[: idx_end + 1],
        "v_pursuer": v_p[: idx_end + 1],
        "distance": distance[: idx_end + 1],
        "los_angle_deg": los_angle_deg[: idx_end + 1],
        "intercepted": intercepted,
        "t_intercept": t_intercept,
        "capture_radius": capture_radius,
    }
    return out

In [ ]:
lead_output = widgets.Output()
lead_status = HTML(value="<b>Lead Pursuit Status:</b> Ready")

lead_k_slider = FloatSlider(description="k_guid", min=0.5, max=8.0, step=0.1, value=3.0, continuous_update=False)
lead_drag_slider = FloatSlider(description="k_drag", min=0.0, max=0.2, step=0.005, value=0.06, continuous_update=False)
lead_turn_slider = FloatSlider(description="turn deg/s", min=8.0, max=100.0, step=1.0, value=40.0, continuous_update=False)
lead_speed_slider = FloatSlider(description="v0", min=0.3, max=3.5, step=0.05, value=1.4, continuous_update=False)
lead_tau_gain_slider = FloatSlider(description="tau_gain", min=0.2, max=2.5, step=0.05, value=1.0, continuous_update=False)

lead_x0_slider = FloatSlider(description="x0", min=-8.0, max=2.0, step=0.2, value=-4.0, continuous_update=False)
lead_y0_slider = FloatSlider(description="y0", min=-4.0, max=4.0, step=0.1, value=-2.0, continuous_update=False)
lead_z0_slider = FloatSlider(description="z0", min=-3.0, max=3.0, step=0.1, value=-0.7, continuous_update=False)

lead_ampy_slider = FloatSlider(description="Ay", min=0.0, max=3.0, step=0.1, value=1.5, continuous_update=False)
lead_ampz_slider = FloatSlider(description="Az", min=0.0, max=2.0, step=0.1, value=0.8, continuous_update=False)
lead_omega_slider = FloatSlider(description="omega", min=0.1, max=1.2, step=0.05, value=0.45, continuous_update=False)

lead_play = widgets.Play(value=0, min=0, max=1, step=1, interval=45, description="Play", disabled=False)
lead_frame = widgets.IntSlider(value=0, min=0, max=1, step=1, description="frame", continuous_update=True)
widgets.jslink((lead_play, "value"), (lead_frame, "value"))

lead_cache = {}


def _render_lead_frame(frame_idx):
    t = lead_cache["t"]
    pt = lead_cache["p_target"]
    pp = lead_cache["p_pursuer"]
    pa = lead_cache["p_aim"]
    tau = lead_cache["tau_lead"]
    dist = lead_cache["distance"]
    los = lead_cache["los_angle_deg"]

    i = int(np.clip(frame_idx, 0, len(t) - 1))

    with lead_output:
        lead_output.clear_output(wait=True)

        fig = plt.figure(figsize=(15.5, 5.4), constrained_layout=True)
        ax3d = fig.add_subplot(1, 3, 1, projection="3d")
        axd = fig.add_subplot(1, 3, 2)
        axa = fig.add_subplot(1, 3, 3)

        ax3d.plot(pt[:, 0], pt[:, 1], pt[:, 2], color="tab:blue", alpha=0.35, linewidth=2.0, label="Target full path")
        ax3d.plot(pp[: i + 1, 0], pp[: i + 1, 1], pp[: i + 1, 2], color="crimson", linewidth=2.5, label="Pursuer path")

        ax3d.scatter([pt[i, 0]], [pt[i, 1]], [pt[i, 2]], color="tab:blue", s=45, label="Target")
        ax3d.scatter([pp[i, 0]], [pp[i, 1]], [pp[i, 2]], color="crimson", s=45, label="Pursuer")

        # Gold star = automatic look-ahead aim point.
        ax3d.scatter(
            [pa[i, 0]],
            [pa[i, 1]],
            [pa[i, 2]],
            color="gold",
            edgecolor="black",
            marker="*",
            s=180,
            linewidth=0.7,
            label="Aim point",
        )

        ax3d.plot(
            [pp[i, 0], pa[i, 0]],
            [pp[i, 1], pa[i, 1]],
            [pp[i, 2], pa[i, 2]],
            color="goldenrod",
            linestyle="--",
            linewidth=1.2,
            alpha=0.9,
            label="Aim line",
        )

        xyz = np.vstack((pt, pp, pa))
        mins = xyz.min(axis=0)
        maxs = xyz.max(axis=0)
        center = 0.5 * (mins + maxs)
        span = max((maxs - mins).max(), 1.0)
        half = 0.55 * span

        ax3d.set_xlim(center[0] - half, center[0] + half)
        ax3d.set_ylim(center[1] - half, center[1] + half)
        ax3d.set_zlim(center[2] - half, center[2] + half)
        ax3d.set_box_aspect([1, 1, 1])
        ax3d.set_title(f"3D Lead-Pursuit (auto tau={tau[i]:.2f} s)")
        ax3d.set_xlabel("x")
        ax3d.set_ylabel("y")
        ax3d.set_zlabel("z")
        ax3d.view_init(elev=22, azim=-62)
        ax3d.legend(loc="upper left")

        axd.plot(t, dist, color="0.7", linewidth=1.8)
        axd.plot(t[: i + 1], dist[: i + 1], color="crimson", linewidth=2.3)
        axd.axvline(t[i], color="black", linestyle="--", linewidth=1.0)
        axd.axhline(lead_cache["capture_radius"], color="tab:green", linestyle=":", linewidth=1.4, label="capture radius")
        axd.set_title("Distance-to-Target")
        axd.set_xlabel("time [s]")
        axd.set_ylabel("distance")
        axd.grid(True, alpha=0.35)
        axd.legend(loc="upper right")

        axa.plot(t, los, color="0.7", linewidth=1.8)
        axa.plot(t[: i + 1], los[: i + 1], color="tab:purple", linewidth=2.3)
        axa.axvline(t[i], color="black", linestyle="--", linewidth=1.0)
        axa.set_title("LOS Alignment Error")
        axa.set_xlabel("time [s]")
        axa.set_ylabel("angle [deg]")
        axa.grid(True, alpha=0.35)

        plt.show()


def _recompute_lead(_=None):
    data = simulate_lead_pursuit(
        k_guidance=lead_k_slider.value,
        k_drag=lead_drag_slider.value,
        max_turn_deg=lead_turn_slider.value,
        speed0=lead_speed_slider.value,
        tau_gain=lead_tau_gain_slider.value,
        p0=(lead_x0_slider.value, lead_y0_slider.value, lead_z0_slider.value),
        y_amp=lead_ampy_slider.value,
        z_amp=lead_ampz_slider.value,
        omega=lead_omega_slider.value,
    )

    lead_cache.clear()
    lead_cache.update(data)

    n = len(lead_cache["t"])
    lead_frame.max = max(n - 1, 1)
    lead_play.max = max(n - 1, 1)
    lead_play.value = 0
    lead_frame.value = 0

    if lead_cache["intercepted"]:
        lead_status.value = f"<b>Lead Pursuit Status:</b> Intercept at t = {lead_cache['t_intercept']:.2f} s"
    else:
        lead_status.value = f"<b>Lead Pursuit Status:</b> No intercept. Final miss distance = {lead_cache['distance'][-1]:.2f}"

    _render_lead_frame(0)


def _on_lead_frame_change(change):
    if change["name"] == "value":
        _render_lead_frame(change["new"])


for s in [
    lead_k_slider,
    lead_drag_slider,
    lead_turn_slider,
    lead_speed_slider,
    lead_tau_gain_slider,
    lead_x0_slider,
    lead_y0_slider,
    lead_z0_slider,
    lead_ampy_slider,
    lead_ampz_slider,
    lead_omega_slider,
]:
    s.observe(_recompute_lead, names="value")

lead_frame.observe(_on_lead_frame_change, names="value")

lead_controls = VBox(
    [
        HTML("<h4>Lead guidance and nonlinear dynamics</h4>"),
        HBox([lead_k_slider, lead_drag_slider, lead_turn_slider, lead_speed_slider, lead_tau_gain_slider]),
        HTML("<h4>Pursuer initial condition</h4>"),
        HBox([lead_x0_slider, lead_y0_slider, lead_z0_slider]),
        HTML("<h4>Target maneuver</h4>"),
        HBox([lead_ampy_slider, lead_ampz_slider, lead_omega_slider]),
        HTML("<h4>Animation</h4>"),
        HBox([lead_play, lead_frame]),
        lead_status,
    ]
)

display(lead_controls, lead_output)
_recompute_lead()